**Building an RAG Pipeline Using PDF Chunking and Retrieval**

Build a a Retrieval-Augmented-Generation Pipeline using various tools in langchain library to process PDF documents , convert them into vectorized chunks and retrieve relevant information using semantic search .
Learnings  - STrengthen understanding on-  real world content


*   Document chunking
*   Embedding
*   Vector search workflow


Relative path for loading the pdfs-
<!--
/content/sample_data/rag_pdfs/beyond-the-copilot-scaling-the-agentic-product-development-life-cycle.pdf

/content/sample_data/rag_pdfs/one-year-of-agentic-ai-six-lessons-from-the-people-doing-the-work_vf.pdf -->


/content/sample_data/cep_data_monetization_problem_11.pdf

/content/sample_data/cep_data_monetization_problem_2.pdf

/content/sample_data/Data_Monetisation.pdf

# Step 1: Install dependencies including pdf and text loaders

In [ ]:
# Import the libraries , all of the tools for agent building
!pip install langchain openai PyPDF2 faiss-cpu tiktoken langchain-openai langchain-classic pypdf
!pip install langchain_community
!pip install langchain_text_splitters
!pip install langchain_classic

  Using cached langchain-1.3.18-py3-none-any.whl.metadata (6.1 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.5 MB/s eta 0:00:00
  Using cached tiktoken-0.14.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.7 kB)
  Using cached langchain_openai-1.6.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
  Using cached langchain_core-1.6.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached langgraph-1.2.11-py3-none-any.whl.metadata (4.9 kB)
  Using cached pydantic-2.13.5-py3-none-any.whl.metadata (110 kB)
  Using cached anyio-4.14.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached numpy-2.5.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.8 MB/s eta 0:00:00
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached certifi-2026.7.22-py3-

# Step 2: Import dependencies

In [ ]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter# for chunking
# vectorstores is for storage purposes  # computers only understand numbers hence vectorize the text, called embeddings store in vector db
# FAISS is one of the classes helping to store the embeddings in a vector db
from langchain_classic.chains import RetrievalQA  # helping to load some documets , load some embeddings
#update

from langchain_community.vectorstores import FAISS # This is for
from langchain_community.document_loaders import TextLoader
# import the libraries for PDFs
from langchain_community.document_loaders import PyPDFLoader
from PyPDF2 import PdfReader

/tmp/ipykernel_3138/1653098350.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader


# Step 3: Set Azure OpenAI credentials

In [ ]:
os.environ["AZURE_OPENAI_API_KEY"] = "2ABecnfxzhRg4M5D6pBKiqxXVhmGB2WvQ0aYKkbTCPsj0JLKsZPfJQQJ99BDAC77bzfXJ3w3AAABACOGi3sC"# same clone of the lab sessions

# Step 4: Load and chunk your custom FAQ document

Preserves semantic boundaries, making chunks safer for embeddings, search, and RAG workflows



In [ ]:
from langchain_core import documents
loader = DirectoryLoader(
    path='/home/subhroy557/ai-rag_agent/',
    glob='*.pdf',
    loader_cls=PyPDFLoader
)

documents = loader.lazy_load()

for document in documents:
    print(document.metadata)

{'producer': 'Adobe PDF library 17.00', 'creator': 'Adobe Illustrator 28.1 (Windows)', 'creationdate': '2024-06-28T12:26:33+06:30', 'moddate': '2024-06-28T12:26:33+05:30', 'title': 'C5_Data_Monetisation_V4', 'source': '/home/subhroy557/ai-rag_agent/Data_Monetisation.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}
{'producer': 'Skia/PDF m154 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'cep_data_monetization_problem_2.docx', 'source': '/home/subhroy557/ai-rag_agent/cep_data_monetization_problem_2.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}
{'producer': 'Skia/PDF m154 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'cep_data_monetization_problem_2.docx', 'source': '/home/subhroy557/ai-rag_agent/cep_data_monetization_problem_2.pdf', 'total_pages': 4, 'page': 1, 'page_label': '2'}
{'producer': 'Skia/PDF m154 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'cep_data_monetization_problem_2.docx', 'sour

Step 5: Split into chucks all of the loaded documents

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)# how do we define the chunk size ? overlap is needed to retain context across chunks

# Re-load the documents because the generator was consumed in the previous cell.
documents = loader.lazy_load()
# Convert the documents generator to a list.
documents_list = list(documents)

docs = text_splitter.split_documents(documents_list)

# Check if the loader is initialized correctly
print(f'Loaded {len(documents_list)} documents')
print(f'Split into {len(docs)} chunks')

Loaded 12 documents
Split into 12 chunks


# Step 5: Create vectorstore using Azure embeddings


In [ ]:
# Embed documnet chunks and store them in an FAISS vector store
embedddings = AzureOpenAIEmbeddings(
    azure_endpoint="https://openai-api-management-gw.azure-api.net",
    deployment="text-embedding-ada-002",
    chunk_size=500,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version="2023-05-15",
    model="text-embedding-ada-002"
)
vectorstore = FAISS.from_documents(docs, embedddings)


# Step 6: Initialize the Azure OpenAI LLM

In [ ]:
# init the azure openai llm for deterministic output , set temperature = 0
llm = AzureChatOpenAI(
    azure_endpoint="https://openai-api-management-gw.azure-api.net",
    api_version="2025-01-01-preview",
    deployment_name="gpt-5-mini"

)
#

\# Step 7: Create the RAG chain

In [ ]:
qa_chain= RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),

    return_source_documents=True
)

# Step 8: Ask a question

In [ ]:
query1= "What examples are existing in the RAG store with respect to Data monetization ? "
result1 = qa_chain.invoke(query1)

# query2= "What are the different types of Data monetization described in the documents ? "
# result2 = qa_chain.invoke(query2)


# Step 9: Print results

In [ ]:
# Print all of the above results
print(result1)
# print(result2)
# print(result3)
# print(result4)

{'query': 'What examples are existing in the RAG store with respect to Data monetization ? ', 'result': 'I don’t have access to your RAG store, so I can’t list its actual items. If you want me to inspect it I’ll need either (a) a brief export/listing of the RAG index or (b) permission/credentials and details about the store (engine, query API).  \n\nUntil then, here are (A) the realistic types of items you would commonly find in a RAG store that relate to data monetization, (B) how those map to direct vs indirect monetization, and (C) practical search queries / metadata filters you can run to locate them quickly.\n\nA — Typical RAG-store artifacts relevant to Data Monetization\n- Data product catalog entries\n  - Descriptions of offered data products (raw feeds, aggregated datasets, APIs), schema, sample records, pricing tiers.\n- Market research / TAM analysis\n  - Reports on buyer demand, competitor offerings, price benchmarks and industry use cases.\n- Legal & privacy reviews\n  - G

In [ ]:
query2= "What are the different types of Data monetization described in the documents ? "
result2 = qa_chain.invoke(query2)
print(result2)

{'query': 'What are the different types of Data monetization described in the documents ? ', 'result': 'The documents describe two high-level types of data monetization:\n\n1) Direct data monetization  \n- Definition: selling, licensing, or otherwise exchanging data or data-driven products externally to generate explicit, measurable revenue.  \n- Examples / approaches mentioned:  \n  - Direct sale of raw, anonymized, aggregated, or insight-based data products to third parties.  \n  - Data-as-a-Service / API access or subscription licensing for data products.  \n  - Packaging analytics or derived data (insights, models) as commercial products or feeds for external customers.\n\n2) Indirect data monetization  \n- Definition: using data to improve internal capabilities, operations, products or decision-making so the organization gains strategic or operational value (not necessarily a direct revenue stream).  \n- Examples / approaches mentioned:  \n  - Strategic decision‑making (e.g., algo

In [ ]:
query3= "What do the customers of Medtronic gain as per the example of Data monetization ? "
result3 = qa_chain.invoke(query3)

print(result3)

{'query': 'What do the customers of Medtronic gain as per the example of Data monetization ? ', 'result': 'In that example Medtronic’s customers (physicians) receive detailed, integrated reports that combine glucose and physical‑activity data — eliminating manual data entry and reducing measurement errors. That enables clinicians to optimize therapy, make data‑driven lifestyle recommendations, and improve patient outcomes.', 'source_documents': [Document(id='9bc7b82d-da6a-4cc7-a7a8-54667872de6b', metadata={'producer': 'Skia/PDF m154 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'cep_data_monetization_problem_1.docx', 'source': '/home/subhroy557/ai-rag_agent/cep_data_monetization_problem_11.pdf', 'total_pages': 7, 'page': 5, 'page_label': '6'}, page_content='●\n \nFitbit:\n \nExpands\n \nfrom\n \nconsumer\n \nhealth\n \ntracking\n \ninto\n \nclinically\n \nvalidated\n \nmedical\n \nuse\n \ncases,\n \nstrengthening\n \nbrand\n \ncredibility\n \nand\n \nincreasin

In [ ]:
query4= "What careful considerations must be made during dmonetization opportunities ? "
result4 = qa_chain.invoke(query4)
# print results
print(result4)

{'query': 'What careful considerations must be made during dmonetization opportunities ? ', 'result': 'When evaluating data‑monetization opportunities you should treat them as strategic business initiatives — not just one‑off transactions. Key considerations span legal, ethical, technical, commercial and operational domains. Below is a compact checklist of what to evaluate and recommended mitigations.\n\nHigh‑level categories\n- Strategic alignment\n  - Ensure the opportunity supports overall business goals (revenue, client experience, risk reduction, competitive positioning).\n  - Define short‑term KPIs and long‑term strategic metrics before you proceed.\n\n- Market & commercial\n  - Market demand and pricing: research buyers, comparable datasets, acceptable formats and price points.\n  - Packaging and productization: how will data be cleaned, standardized, enriched and delivered (APIs, feeds, dashboards)?\n  - Cost–benefit and ROI: include collection, normalization, storage, integrat